In [96]:
import os
import glob
import pandas as pd

In [97]:
# Augmented UNet statistics

In [98]:
synthetic_images_count = pd.read_csv('synthetic_images_count.csv')

In [99]:
synthetic_images_count['Count of Real Images'] = len(os.listdir('train'))

In [100]:
synthetic_images_count['Actual Synthetic to Real Ratio'] = synthetic_images_count['Count of Synthetic Images Added'] / synthetic_images_count['Count of Real Images']

In [101]:
for col in ['Discriminator Final Real Accuracy', 'Discriminator Final Synthetic Accuracy', 'Discriminator Final Train Accuracy']:
    synthetic_images_count[col] = 0

In [102]:
synthetic_images_count

,Generative Model,Synthetic to Real Ratio,Count of Synthetic Images Added,Count of Real Images,Actual Synthetic to Real Ratio,Discriminator Final Real Accuracy,Discriminator Final Synthetic Accuracy,Discriminator Final Train Accuracy
0,singan-seg,9.0,3708,6870,0.539738,0,0,0
1,singan-seg,5.0,1374,6870,0.200000,0,0,0
2,singan-seg,1.0,274,6870,0.039884,0,0,0
3,singan-seg,0.5,136,6870,0.019796,0,0,0


In [105]:
import re
import ast
import pandas as pd

def parse_metric_log(log_str: str) -> dict:
    """
    Parse a multi-line metric string of the form:
        Best Epoch: 1
        Dice Coefficient: 0.5113
        Best Train Loss: 0.3366
        Best Val Loss: 0.5642
        Train Loss History: [0.33657772093377336]
        Val Loss History: [0.5642358295073083]

    Returns a dict mapping each field name to its (int, float, or list) value.
    """
    metrics = {}
    for line in log_str.strip().splitlines():
        key, val = line.split(":", 1)
        val = val.strip()
        if key.endswith("History"):
            # parse the bracketed list into a Python list of floats
            metrics[key] = ast.literal_eval(val)
        else:
            # convert to int if no decimal point, else float
            metrics[key] = int(val) if val.isdigit() else float(val)
    return metrics

In [124]:
import numpy as np

In [ ]:
for i in synthetic_images_count.index:
    folder = f'unet_performance_statistics/model-{synthetic_images_count.loc[i, 'Generative Model']}-synthetic-real-ratio-{synthetic_images_count.loc[i, 'Synthetic to Real Ratio']}*'
    sorted_folders = sorted(glob.glob(folder))[::-1]
    for folder in sorted_folders:
        best_dice_epoch_path = os.path.join(folder, 'best_dice_epoch.txt')
        discriminator_statistics_path = os.path.join(folder, 'discriminator_statistics.csv')
        if os.path.exists(best_dice_epoch_path) and os.path.exists(discriminator_statistics_path):
            with open(best_dice_epoch_path, 'r') as f:
                best_dice_epoch = f.read().strip()

            training_export_data = parse_metric_log(best_dice_epoch)

            search_query = 'Dice Coefficient: '
            synthetic_images_count.loc[i, 'Validation Accuracy (Dice Coefficient)'] = training_export_data['Dice Coefficient']
            synthetic_images_count.loc[i, 'Best Training Loss'] = training_export_data['Best Train Loss']
            synthetic_images_count.loc[i, 'Best Validation Loss'] = training_export_data['Best Val Loss']
            synthetic_images_count.loc[i, 'Train Loss History'] = np.array(training_export_data['Train Loss History'])
            synthetic_images_count.loc[i, 'Validation Loss History'] = np.array(training_export_data['Val Loss History'])
            
            discriminator_statistics = pd.read_csv(discriminator_statistics_path)

            synthetic_images_count.loc[i, 'Discriminator Final Real Accuracy'] = discriminator_statistics['discriminator_train_real_accuracy'].values[-1]
            synthetic_images_count.loc[i, 'Discriminator Final Synthetic Accuracy'] = discriminator_statistics['discriminator_train_fake_accuracy'].values[-1]
            synthetic_images_count.loc[i, 'Discriminator Final Train Accuracy'] = discriminator_statistics['discriminator_train_accuracy'].values[-1]
            break

In [129]:
synthetic_images_count

,Generative Model,Synthetic to Real Ratio,Count of Synthetic Images Added,Count of Real Images,Actual Synthetic to Real Ratio,Discriminator Final Real Accuracy,Discriminator Final Synthetic Accuracy,Discriminator Final Train Accuracy,Validation Accuracy,Best Training Loss,Best Validation Loss,Train Loss History,Validation Loss History
0,singan-seg,9.0,3708,6870,0.539738,0.998107,0.999730,0.998676,0.5113,0.3366,0.5642,0.336578,0.564236
1,singan-seg,5.0,1374,6870,0.200000,0.998835,0.997817,0.998666,0.4354,0.4496,0.5363,0.449578,0.536312
2,singan-seg,1.0,274,6870,0.039884,0.998835,0.952555,0.997060,0.6180,0.4084,0.4384,0.408389,0.438380
3,singan-seg,0.5,136,6870,0.019796,0.999854,0.867647,0.997288,0.4366,0.4124,1.1325,0.412447,1.132503


In [ ]:
# Vanilla UNet statistics

In [86]:
row = {
    'Generative Model': 'none',
    'Synthetic to Real Ratio': 0,
    'Count of Synthetic Images Added': 0,
    'Count of Real Images': len(os.listdir('train')),
    'Actual Synthetic to Real Ratio': 0,

    'Validation Accuracy': 0.5902,

    'Discriminator Final Real Accuracy': 0,
    'Discriminator Final Synthetic Accuracy': 0,
    'Discriminator Final Train Accuracy': 0
}

In [91]:
synthetic_images_count = pd.concat([synthetic_images_count, pd.DataFrame([row])])

In [92]:
synthetic_images_count

,Generative Model,Synthetic to Real Ratio,Count of Synthetic Images Added,Count of Real Images,Actual Synthetic to Real Ratio,Discriminator Final Real Accuracy,Discriminator Final Synthetic Accuracy,Discriminator Final Train Accuracy,Validation Accuracy
0,singan-seg,9.0,3708,6870,0.539738,0.998112,0.999563,0.999055,0.4522
1,singan-seg,5.0,1374,6870,0.200000,0.995633,1.000000,0.999272,0.4484
2,singan-seg,1.0,274,6870,0.039884,0.934307,0.998544,0.996080,0.4652
3,singan-seg,0.5,136,6870,0.019796,0.897059,0.998835,0.996859,0.4648
0,none,0.0,0,6870,0.000000,0.000000,0.000000,0.000000,0.5902
